In [ ]:
#Libs
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from selenium.common.exceptions import UnexpectedAlertPresentException, NoAlertPresentException
from datetime import datetime
from turtle import onclick
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote
from pathlib import Path
from tqdm import tqdm
import os, re, mimetypes, time, unicodedata, win32com.client, requests, urllib3, pandas as pd, openpyxl
import tempfile  
import subprocess
from urllib.parse import urljoin
import shutil
import winsound 
import traceback
import json
import pandas as pd

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [ ]:
# ===== CONFIGURAÇÃO =====
excel_path = r"./DA Migration Índice de Conteúdos - ClassRoom.xlsx"


URL_LOGIN = "https://gehealthcare.sistematutor.com.br/tutor/sistema/login.do?method=logout"
EMAIL = "gabriel.santos@gehealthcare.com"
SENHA = "Golias123@"

# ===== CHROME =====
chrome_options = Options()

# User-Agent realista
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

chrome_options.add_experimental_option('prefs', {
    'profile.password_manager_enabled': False,
    'credentials_enable_service': False,
    'profile.default_content_setting_values.notifications': 2,
    'download.prompt_for_download': False,
    'download.directory_upgrade': True,
    'safebrowsing.enabled': True,
    'profile.default_content_setting_values.automatic_downloads': 1,
})
chrome_options.add_experimental_option('excludeSwitches', ['enable-logging', 'enable-automation', 'enable-sync'])
chrome_options.add_experimental_option('useAutomationExtension', False)

# Anti-detection arguments
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument('--disable-notifications')
chrome_options.add_argument('--disable-infobars')
chrome_options.add_argument('--disable-popup-blocking')
chrome_options.add_argument('--disable-save-password-bubble')
chrome_options.add_argument('--disable-features=IsolateOrigins,site-per-process,TranslateUI')
chrome_options.add_argument('--disable-site-isolation-trials')
chrome_options.add_argument('--no-first-run')
chrome_options.add_argument('--no-service-autorun')
chrome_options.add_argument('--disable-default-apps')
chrome_options.add_argument('--disable-sync')
chrome_options.add_argument('--disable-translate')
chrome_options.add_argument('--disable-extensions')
chrome_options.add_argument('--disable-background-timer-throttling')
chrome_options.add_argument('--disable-client-side-phishing-detection')
chrome_options.add_argument('--disable-component-extensions-with-background-pages')
chrome_options.add_argument('--disable-hang-monitor')
chrome_options.add_argument('--disable-prompt-on-repost')
chrome_options.add_argument('--metrics-recording-only')

chrome_options.page_load_strategy = 'normal'

# Inicializar driver e wait globais
driver = webdriver.Chrome(options=chrome_options)
driver.set_page_load_timeout(30)
driver.set_script_timeout(30)
driver.maximize_window()


# Inicializar WebDriverWait global
wait = WebDriverWait(driver, 30)

In [ ]:
# Acessar site e realizar login
def iniciar_login(URL_LOGIN: str, EMAIL: str, SENHA: str, timeout: int = 30):
    if driver is None:
        raise RuntimeError("O driver global não foi inicializado. Execute a Célula 1 primeiro.")
    _wait = WebDriverWait(driver, timeout)

    print("✓ Iniciando processo de login...")
    driver.get(URL_LOGIN)
    time.sleep(2)

    campo_email = _wait.until(EC.presence_of_element_located((By.ID, "login")))
    campo_email.clear()
    campo_email.send_keys(EMAIL)
    print("✓ Email inserido!")

    campo_senha = driver.find_element(By.NAME, "senha")
    campo_senha.clear()
    campo_senha.send_keys(SENHA)
    print("✓ Senha inserida!")

    botao_login = driver.find_element(By.XPATH, "//button[@type='submit']")
    botao_login.click()
    print("✓ Login clicado!")
    time.sleep(3)

    def buscar_codigo_outlook():
        print("Aguardando 30 s para recebimento do código...")
        time.sleep(30)
        outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
        inbox = outlook.GetDefaultFolder(6)
        messages = inbox.Items
        messages.Sort("[ReceivedTime]", True)
        for message in list(messages)[:10]:
            try:
                if "CÓDIGO DE VALIDAÇÃO" in str(message.Subject).upper():
                    body = str(message.Body)
                    codigo_match = re.search(r'Código de ativação:\s*(\d{6})', body, re.IGNORECASE)
                    if codigo_match:
                        return codigo_match.group(1)
            except Exception:
                continue
        return None

    codigo = buscar_codigo_outlook()
    if not codigo:
        raise ValueError("Não foi possível encontrar o código de validação no Outlook.")
    print(f"✓ Código obtido: {codigo}")

    campo_codigo = _wait.until(EC.presence_of_element_located((By.ID, "code")))
    campo_codigo.clear()
    campo_codigo.send_keys(codigo)
    print("✓ Código inserido!")

    botao_salvar = _wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@onclick='salvar()']")))
    botao_salvar.click()
    print("✓ Logado com sucesso!")
    time.sleep(3)


In [ ]:

# ── Utilitário: detectar idioma pelo Produto 1 ──────────────────────────────
def detectar_idioma(produto1: str) -> str:
    """
    Retorna 'ESP' ou 'PT' com base no conteúdo da coluna Produto 1.
    Regras:
      - Contém 'ESP'   → ESP  (Site Espanhol, value="2")
      - Contém 'IMSS'  → ESP
      - Contém 'PT'    → PT   (Site Português, value="1")
      - Contém 'Fleury'→ PT
      - Padrão         → PT
    """
    p = str(produto1).upper()
    if "IMSS" in p:
        return "ESP"
    if "ESP" in p:
        return "ESP"
    if "FLEURY" in p:
        return "PT"
    if " PT" in p or p.endswith("PT") or "(PT)" in p:
        return "PT"
    return "PT"  # padrão seguro


def carregar_base(nome_arquivo: str = "Acessos Plataforma.xlsx") -> pd.DataFrame:
    """
    Carrega o arquivo Excel da pasta atual.
    Filtra apenas linhas onde 'Acesso Liberado?' == 1 e e-mail não é nulo.
    Remove duplicatas de e-mail (mantém a primeira ocorrência).
    """

    # Usa diretório atual (compatível com notebook e .py)
    caminho = os.path.join(os.getcwd(), nome_arquivo)

    df = pd.read_excel(caminho)

    # Normalizar nomes de colunas (remover espaços extras)
    df.columns = df.columns.str.strip()

    # Filtrar apenas acessos liberados com e-mail válido
    df = df[df["Acesso Liberado?"].isna()].copy()
    df = df[df["E-mail"].notna() & (df["E-mail"].str.strip() != "")].copy()

    # Remover duplicatas de e-mail
    df = df.drop_duplicates(subset=["E-mail"], keep="first").reset_index(drop=True)

    # Criar coluna de idioma derivada do Produto 1
    df["Idioma"] = df["Produto 1"].apply(detectar_idioma)

    print(f"✓ Base carregada: {len(df)} usuários únicos.")
    return df


def aceitar_alerta_se_existir(driver, timeout=3):
    """Aceita alerta do Chrome/Selenium se aparecer e retorna o texto."""
    try:
        WebDriverWait(driver, timeout).until(EC.alert_is_present())
        alert = driver.switch_to.alert
        texto = alert.text
        alert.accept()
        print(f"⚠️ Alerta aceito: {texto}")
        time.sleep(1)
        return texto
    except TimeoutException:
        return None
    except NoAlertPresentException:
        return None


# ── Verificar e matricular ──────────────────────────────────────────────────
def verificar_e_matricular(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada usuário no DataFrame, verifica se já possui matrícula.
    - Matriculado   → marca 'Matriculado'
    - Sem matrícula → realiza o cadastro completo

    Colunas esperadas no df:
        'Nome', 'E-mail', 'Produto 1', 'Instituição', 'GON'
    """

    URL_MATRICULA = "https://gehealthcare.sistematutor.com.br/tutor/sistema/aluno.do?method=pesquisar"

    if driver is None:
        raise RuntimeError("O driver global não foi inicializado. Execute a Célula 1 primeiro.")

    _wait = WebDriverWait(driver, 30)

    if "Status Matrícula" not in df.columns:
        df["Status Matrícula"] = ""

    total = len(df)
    for contador, (idx, row) in enumerate(df.iterrows(), start=1):
        def _val(col):
            if col not in row.index:
                return ""
            v = row[col]
            try:
                if pd.isna(v):
                    return ""
            except Exception:
                pass
            s = str(v).strip()
            if s.lower() in ("nan", "none", "nat"):
                return ""
            return s

        def _sanitize_email(e: str) -> str:
            if not e:
                return ""
            # remove TODOS os espaços (inclusive internos tipo "...com. br")
            e = re.sub(r"\s+", "", e)
            # remove pontuação no fim (.,;: etc.)
            e = re.sub(r"[\.,;:]+$", "", e)
            return e.strip().lower()

        email       = _sanitize_email(_val("E-mail"))
        nome        = _val("Nome")
        instituicao = _val("Instituição")
        gon         = _val("GON")
        idioma      = row["Idioma"]  # 'ESP' ou 'PT'

        print(f"\n{'='*60}")
        print(f"[{contador}/{total}] Processando: {email} | Idioma: {idioma}")

        # ── Etapa 1: pesquisar usuário ──────────────────────────────
        # Garante que nenhum alerta pendente trave a navegação
        aceitar_alerta_se_existir(driver, timeout=1)

        try:
            driver.get(URL_MATRICULA)
            time.sleep(2)
        except UnexpectedAlertPresentException:
            aceitar_alerta_se_existir(driver, timeout=3)
            driver.get(URL_MATRICULA)
            time.sleep(2)

        try:
            campo_pesquisa = _wait.until(EC.presence_of_element_located((By.ID, "pesquisar")))
            campo_pesquisa.clear()
            campo_pesquisa.send_keys(email)
            campo_pesquisa.send_keys(Keys.ENTER)
            time.sleep(3)
        except Exception as e:
            print(f"✗ Erro ao pesquisar {email}: {e}")
            df.at[idx, "Status Matrícula"] = "Erro - Pesquisa"
            continue

        # ── Etapa 2: verificar tabela ───────────────────────────────
        try:
            tabela = _wait.until(EC.presence_of_element_located((By.ID, "tabelaAlunos")))
            linhas = tabela.find_elements(By.CSS_SELECTOR, "tbody tr")

            matriculado = False
            for linha in linhas:
                colunas = linha.find_elements(By.TAG_NAME, "td")
                if len(colunas) >= 5:
                    email_tabela = colunas[4].text.strip().lower()
                    if email_tabela == email.lower():
                        matriculado = True
                        break

            if matriculado:
                print(f"✓ Já possui matrícula: {email}")
                df.at[idx, "Status Matrícula"] = "Matriculado"
                continue

        except Exception as e:
            print(f"⚠ Erro ao verificar tabela para {email}: {e}")
            df.at[idx, "Status Matrícula"] = "Erro - Verificação"
            continue

        # ── Etapa 3: incluir novo aluno ─────────────────────────────
        print(f"→ Sem matrícula. Iniciando cadastro: {email}")

        try:
            botao_incluir = _wait.until(EC.element_to_be_clickable(
                (By.XPATH, "//input[@value='Incluir um novo Aluno']")
            ))
            botao_incluir.click()
            time.sleep(2)

            def preencher_campo(localizador, valor, label, obrigatorio=False, timeout=8):
                """Preenche um campo de forma tolerante.
                - valor vazio  -> skip
                - campo ausente -> skip (a menos que obrigatório)
                - erros não interrompem o cadastro
                """
                if not valor:
                    print(f"  ⏭ {label}: sem valor, pulando.")
                    return False
                try:
                    el = WebDriverWait(driver, timeout).until(
                        EC.presence_of_element_located(localizador)
                    )
                    try:
                        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                    except Exception:
                        pass
                    try:
                        el.clear()
                    except Exception:
                        pass
                    el.send_keys(valor)
                    print(f"  ✓ {label} inserido!")
                    return True
                except Exception as e:
                    if obrigatorio:
                        raise
                    print(f"  ⏭ {label}: campo indisponível, pulando ({type(e).__name__}).")
                    return False

            # Ordem correta: Nome -> E-mail -> Instituição -> scroll -> GON -> Site

            # Nome (opcional - pula se vazio)
            preencher_campo((By.NAME, "nome"), nome, "Nome", obrigatorio=False)

            # E-mail (obrigatório)
            preencher_campo((By.NAME, "email"), email, "E-mail", obrigatorio=True)
            # Alguns campos disparam validação de domínio onblur -> aceitar e seguir
            aceitar_alerta_se_existir(driver, timeout=2)

            # Instituição (campoExtra7) - opcional
            preencher_campo((By.ID, "campoExtra7"), instituicao, "Instituição", obrigatorio=False)
            aceitar_alerta_se_existir(driver, timeout=1)

            # Scroll até o final da página
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)

            # GON (campoExtra10) - opcional
            preencher_campo((By.ID, "campoExtra10"), gon, "GON", obrigatorio=False)
            aceitar_alerta_se_existir(driver, timeout=1)

            # Selecionar site: ESP → value="2" | PT → value="1"
            valor_site = "2" if idioma == "ESP" else "1"
            select_site = _wait.until(EC.presence_of_element_located((By.ID, "unidadeNaoSelecionada")))
            opcao = select_site.find_element(By.XPATH, f".//option[@value='{valor_site}']")
            opcao.click()
            ActionChains(driver).double_click(opcao).perform()
            print(f"  ✓ Site selecionado: {'Espanhol' if idioma == 'ESP' else 'Português'}")
            time.sleep(1)

            # Salvar
            botao_salvar = _wait.until(EC.element_to_be_clickable((By.ID, "btnSalvar1")))
            botao_salvar.click()
            time.sleep(1)

            # Caso exista homônimo, o sistema abre um alerta perguntando se confirma.
            # Regra desejada: clicar OK, salvar novamente e clicar OK no próximo aviso.
            texto_alerta = aceitar_alerta_se_existir(driver, timeout=3)

            if texto_alerta and "mesmo nome" in texto_alerta.lower():
                print("  ⚠ Homônimo encontrado. Confirmando e salvando novamente...")
                botao_salvar = _wait.until(EC.element_to_be_clickable((By.ID, "btnSalvar1")))
                botao_salvar.click()
                time.sleep(1)

            # Aceita o alerta final de confirmação/sucesso, se aparecer
            texto_alerta_final = aceitar_alerta_se_existir(driver, timeout=5)

            print("  ✓ Aluno salvo com sucesso!")
            df.at[idx, "Status Matrícula"] = "Matriculado Agora"
            if texto_alerta:
                df.at[idx, "Detalhe Matrícula"] = texto_alerta
            if texto_alerta_final:
                df.at[idx, "Detalhe Matrícula Final"] = texto_alerta_final

        except Exception as e:
            print(f"  ✗ Erro ao matricular {email}: {e}")
            df.at[idx, "Status Matrícula"] = "Erro - Cadastro"

    print(f"\n{'='*60}")
    print("✓ Processamento concluído!")
    print(df["Status Matrícula"].value_counts().to_string())
    return df

def fechar_alerta_se_existir(driver, aceitar=False):
    try:
        alert = driver.switch_to.alert
        texto = alert.text

        if aceitar:
            alert.accept()
            acao = "aceito"
        else:
            alert.dismiss()
            acao = "cancelado"

        print(f"⚠️ Alerta {acao}: {texto}")
        return texto

    except NoAlertPresentException:
        return None

In [ ]:
print("\n=== [1] Iniciando login no sistema ===")
iniciar_login(URL_LOGIN, EMAIL, SENHA)

# Carregar base e processar
df = carregar_base("Acessos Plataforma.xlsx") 
df_resultado = verificar_e_matricular(df)

# Salvar resultado
df_resultado.to_excel("Resultado_Matriculas.xlsx", index=False)